# Python Executor - Attempt #1

**Stage:** execution_failed
**Created:** 2025-09-04 15:17:54

## Error Context

**Error:** Python execution failed: cannot read execution metadata (Python execution error: No module named 'tomopy')

**Full Traceback:**
```
Traceback (most recent call last):
  File "/Users/magarces/agenticAI_bolt-main/version-control/bolt-4/alpha_berkeley/interfaces/CLI/../../src/framework/services/python_executor/executor_node.py", line 161, in _execute_with_subprocess
    raise CodeRuntimeError(
framework.services.python_executor.exceptions.CodeRuntimeError: Python execution error: No module named 'tomopy'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/Users/magarces/agenticAI_bolt-main/version-control/bolt-4/alpha_berkeley/interfaces/CLI/../../src/framework/services/python_executor/executor_node.py", line 467, in executor_node
    execution_result = await executor.execute_code(
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/magarces/agenticAI_bolt-main/version-control/bolt-4/alpha_berkeley/interfaces/CLI/../../src/framework/services/python_executor/executor_node.py", line 57, in execute_code
    return await self._execute_with_subprocess(wrapped_code, execution_folder)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/magarces/agenticAI_bolt-main/version-control/bolt-4/alpha_berkeley/interfaces/CLI/../../src/framework/services/python_executor/executor_node.py", line 197, in _execute_with_subprocess
    raise CodeRuntimeError(
framework.services.python_executor.exceptions.CodeRuntimeError: Python execution failed: cannot read execution metadata (Python execution error: No module named 'tomopy')

```

**Execution Stage:** Code execution failed during wrapped code execution

**Debug Information:**
- Error Type: CodeRuntimeError
- Error Message: Python execution failed: cannot read execution metadata (Python execution error: No module named 'tomopy')
- Generated Code Length: 1367 characters




In [ ]:

# Load execution context
from framework.context import load_context
context = load_context('../context.json')


In [ ]:
import os
from bluesky import RunEngine
from bluesky.plans import scan, grid_scan
from bluesky.preprocessors import run_wrapper
import numpy as np
import tomopy
import dxchange

def reconstruct_object(folder_path):
    """
    Bluesky plan to reconstruct an object from projection data
    """
    # Initialize results dictionary
    results = {}

    # Detect projection files in the specified folder
    proj_files = [f for f in os.listdir(folder_path) if f.endswith('.tiff') or f.endswith('.tif')]
    proj_files.sort()

    # Load projection data
    proj_data = dxchange.read_tiff_stack(os.path.join(folder_path, proj_files[0]), 
                                         range(len(proj_files)))

    # Define reconstruction parameters
    theta = np.linspace(0, np.pi, len(proj_files))

    # Perform tomographic reconstruction using tomopy
    recon = tomopy.recon(proj_data, theta, algorithm='gridrec')
    
    # Normalize reconstruction
    recon = tomopy.normalize(recon)

    # Save results
    results['reconstruction'] = recon
    results['num_projections'] = len(proj_files)
    results['reconstruction_algorithm'] = 'gridrec'

    return results

def run_reconstruction():
    """
    Execute reconstruction plan
    """
    RE = RunEngine({})
    folder_path = 'test_4'
    results = RE(run_wrapper(reconstruct_object(folder_path)))
    return results